# 日本語文章の分類

In [1]:
!pip install transformers[ja] fugashi ipadic

In [2]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [1]:
from google.colab import drive
import glob
import os
import csv
import torch
from sklearn.model_selection import train_test_split
from transformers import BertForSequenceClassification, BertJapaneseTokenizer
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from transformers import Trainer, TrainingArguments
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='Parameter `function`=*')


In [2]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# データの取得
path = "/content/drive/My Drive/BERT/text/"

dir_files = os.listdir(path=path)
dirs = [f for f in dir_files if os.path.isdir(os.path.join(path, f))]

text_label_data = [] #文章のラベルセット
dir_count = 0        #ディレクトリ数のカウント
file_count = 0       #ファイル数のカウント

for i in range(len(dirs)):
    dir = dirs[i]
    files = glob.glob(path + dir + "/*.txt")
    dir_count += 1

    for file in files:
        if os.path.basename(file) == "LICENSE.txt":
            continue

        with open(file, "r") as f:
            text = f.readlines()[3:]
            text = "".join(text)
            # 文章の前処理
            text = text.translate(str.maketrans({"\n":"", "\t":"", "\r":"", "\u3000":""}))
            text_label_data.append([text, i])

        file_count += 1
        print("\rfiles: " + str(file_count) + "dirs: " + str(dir_count), end="")


files: 7367dirs: 9

In [4]:
# データ分割と保存
news_train, news_test = train_test_split(text_label_data, shuffle=True)
news_path = "/content/drive/My Drive/BERT/"

with open(news_path+"news_train.csv", "w") as f:
    writer = csv.writer(f)
    writer.writerows(news_train)

with open(news_path+"news_test.csv", "w") as f:
    writer = csv.writer(f)
    writer.writerows(news_test)

In [5]:
# model, Tokenizerの読み込み
sc_model = BertForSequenceClassification.from_pretrained("cl-tohoku/bert-base-japanese-whole-word-masking", num_labels=9)
sc_model.cuda()
tokenizer = BertJapaneseTokenizer.from_pretrained("cl-tohoku/bert-base-japanese-whole-word-masking")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-whole-word-masking
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missi

In [6]:
# 文章のトーク数の確認

def tokenize_for_check(batch):
    return tokenizer(batch["text"])

news_path = "/content/drive/My Drive/BERT/"

train_data = load_dataset("csv", data_files=news_path+"news_train.csv", column_names=["text", "label"], split="train")
train_data_checked = train_data.map(tokenize_for_check, batched=True, batch_size=len(train_data))


token_lengths = [len(ids) for ids in train_data_checked["input_ids"]]


Generating train split: 0 examples [00:00, ? examples/s]

Parameter 'function'=<function tokenize_for_check at 0x7d10d62df240> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/5525 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2033 > 512). Running this sequence through the model will result in indexing errors


In [7]:
print("最大トークン数:", max(token_lengths))
print("平均のトークン数:", round(np.mean(token_lengths)))
print("中央値:", np.median(token_lengths))
print("95%の文章が収まる長さ   :", np.percentile(token_lengths, 95))

最大トークン数: 4903
平均のトークン数: 742
中央値: 620.0
95%の文章が収まる長さ   : 1526.4000000000005


In [8]:
# データの読み込み

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=256)


train_data = train_data.map(tokenize, batched=True, batch_size=len(train_data))
train_data.set_format("torch", columns=["input_ids", "label"])

test_data = load_dataset("csv", data_files=news_path+"news_test.csv", column_names=["text", "label"], split="train")
test_data = test_data.map(tokenize, batched=True, batch_size=len(test_data))
test_data.set_format("torch", columns=["input_ids", "label"])

Map:   0%|          | 0/5525 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1842 [00:00<?, ? examples/s]

In [9]:
# 評価用の関数の設定

def compute_metrics(result):
    labels = result.label_ids #正解ラベル
    preds = result.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
    }

In [10]:
# ハイパーパラメーターの設定
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 2,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 32,
    warmup_steps = 500,
    weight_decay = 0.01,
    logging_dir = "./logs",
    eval_strategy = "steps"
)

trainer = Trainer(
    model = sc_model,
    args = training_args,
    compute_metrics = compute_metrics,
    train_dataset = train_data,
    eval_dataset = test_data
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
500,1.044406,0.536469,0.855049
1000,0.315552,0.359390,0.911509
1382,0.315552,0.267073,0.932682


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1382, training_loss=0.5518347357875532, metrics={'train_runtime': 664.8588, 'train_samples_per_second': 16.62, 'train_steps_per_second': 2.079, 'total_flos': 1453779945446400.0, 'train_loss': 0.5518347357875532, 'epoch': 2.0})

In [12]:
# モデル評価
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
0.315552,0.267073,1382,0.932682


{'eval_loss': 0.26707321405410767, 'eval_accuracy': 0.9326818675352877}

In [13]:
predictions = trainer.predict(test_data)
y_pred = predictions.predictions.argmax(axis=1)

In [14]:
y_true = np.array(test_data["label"])

In [15]:
wrong_idx = np.where(y_pred != y_true)[0]

In [16]:
raw_test_data = test_data.with_format(None)

In [17]:
for i in wrong_idx[:10]:
    idx = int(i)

    print("="*80)
    print(f"【インデックス: {idx}】")
    print("ニュース本文:")

    print(raw_test_data["text"][idx][:300])

    print("\n正解ラベル:")
    print(dirs[int(y_true[idx])])

    print("\n予測ラベル:")
    print(dirs[int(y_pred[idx])])

【インデックス: 11】
ニュース本文:
総務省統計局の調査によればパソコンの普及率は2005年に80%を超えたという。もはやパソコンは私達の生活に欠かせない家電の一つとも言える。記録メディアとしての機能も持ち合わせているため、使い勝手が良く、家族との思い出の写真から、ダウンロードした流行りの音楽や動画などをパソコンに保存している方は多い。しかし、使い勝手が良いだけに、万が一の事態でパソコンが壊れてしまったりするとこれまでの思い出の写真やせっかく購入した音楽が全て消えてしまうようなことだって起こりかねない。また、データを大量に保存しすぎてパソコンの動作が鈍くなってしまったと感じる方も多いのではないだろうか。そんな中、パソコンソフト販売

正解ラベル:
livedoor-homme

予測ラベル:
it-life-hack
【インデックス: 12】
ニュース本文:
気がつけば今年も残すところあと1ヶ月ちょっと。街の景色も少しずつクリスマスカラーに彩られ始めていて、今年はどんな風に過ごそうか、と考えを巡らせる時期ですよね。なかには今や定番となった女子会で過ごそうという人も多いのでは？ ちょっとお得な女子会プランのあるお店で気の合う女子とワイワイ過ごすクリスマス、誰かのお家に集まって、尽きないガールズトークに華を咲かせながらほっこり過ごすクリスマスもいいけれど、今年は“オトナ”のあなたにふさわしいラグジュアリーで特別な一夜を過ごしてみませんか？ リムジンのお出迎えにショッピング、サロンでキレイになった後は贅沢な食事を囲んでゆっくりとディナー、シャンパングラス

正解ラベル:
dokujo-tsushin

予測ラベル:
peachy
【インデックス: 16】
ニュース本文:
2011年4月30日（土）浅野忠信主演『これでいいのだ！！映画☆赤塚不二夫』の映画化を記念し、東映ビデオ株式会社が赤塚アニメコンピレーションDVDを制作、2011年5月21日（土）を発売することになった。今回DVD化される作品は、東映まんがまつり、東宝チャンピオンまつりで公開された劇場版とテレビで放送された「もーれつア太郎」「ひみつのアッコちゃん」「天才バカボン」「おそ松くん」から抜粋した13話のアニメーションが一つになった傑作、珍作など多数。なんとこの「赤塚アニメコンピレーションDVD」に収録された

In [18]:
news_path = "/content/drive/My Drive/BERT/"

sc_model.save_pretrained(news_path)
tokenizer.save_pretrained(news_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/My Drive/BERT/tokenizer_config.json',
 '/content/drive/My Drive/BERT/vocab.txt',
 '/content/drive/My Drive/BERT/added_tokens.json')

In [19]:
loaded_model = BertForSequenceClassification.from_pretrained(news_path)
loaded_model.cuda()
loaded_tokenizer = BertJapaneseTokenizer.from_pretrained(news_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 日本語ニューズの分類

In [20]:
# 読み込んだモデルを使ってニュースの分類
# 分類したいニュースの読み込み

category = "movie-enter"
sample_path = "/content/drive/My Drive/BERT/text/"
files = glob.glob(sample_path + category + "/*.txt")
file = files[42] #適当に選定

dir_files = os.listdir(path=sample_path)
dirs = [f for f in dir_files if os.path.isdir(os.path.join(sample_path, f))]


# 前処理
with open(file, "r") as f:
    sample_text = f.readlines()[3:]
    sample_text = "".join(sample_text)
    sample_text = sample_text.translate(str.maketrans({"\n":"", "\t":"", "\r":"", "\u3000":""}))

print(sample_text)

ＳＦ小説の巨匠ロバート・Ａ・ハインラインの傑作『宇宙の戦士』を映画化し、巨大昆虫と戦う兵士たちの凄まじい生死をかけた戦いを描いた『スターシップ・トゥルーパーズ』シリーズ。これまで、シリーズ作品が第三作まで製作され、クエンティン・タランティーノを始めとした世界中のファンから熱烈な支持を受けている。この人気シリーズの最新作『スターシップ・トゥルーパーズ：インベイジョン』が、誕生15周年を記念し日本から世界に向けて発信される。これまでに予告編や場面写真が公開されており、第一作からの主人公、ジョニー・リコが“将軍”として登場することや、女性パイロットのカルメン、サイキックの独善主義者カールといったおなじみのキャラクターの再結集、シリーズを通して注目の的であったパワードスーツ“マローダー”が遂に大活躍することなどが判明しており、ファンの期待は高まるばかり。そんな『スターシップ・トゥルーパーズ』ファンの胸をさらに熱くさせるキャンペーンがはじまった。その名も“『スターシップ・トゥルーパーズ インベイジョン』バグ撲滅Twitterキャンペーン”。 このキャンペーンはシリーズ劇中でおなじみのプロパガンダ映像「地球連邦軍ニュース」をモチーフにしたものだ。劇中の「地球連邦ニュース」では地球人と兵士向けへのプロパガンダ映像の最後に「もっと情報を？」という表示が現れ、クリックすると新しい映像が現れるという演出がある。これにちなみ今回のキャンペーンでは、ハッシュタグ「#StiBugs」をつけてツイートし、そのつぶやきが一定数に達することで、新しい「地球連邦ニュース」映像を公式サイトで閲覧することが可能になるというもの。公式サイトのボタンを押して呟くだけで気軽にトルーパー（兵士）たちの戦いを後押しできる。なお、第一弾ニュース映像は111ツイート集まると視聴することができ、合計3種類の「地球連邦ニュース」が用意されているとのことだ。詳細はキャンペーンサイトで確認できる。さあ、野郎ども、今日は死に日和だ。バグにツイートをくれてやれ！『スターシップ・トルーパーズ：インベイジョン』は7月21日より新宿ピカデリー他 全国ロードショー『スターシップ・トルーパーズ：インベイジョン』 - キャンペーンページ『スターシップ・トルーパーズ：インベイジョン』 - 作品情報【関連記事】・死ぬまで戦え！『スターシップ・ト

In [21]:
# モデル挿入への変換
max_lenght = 512
words = loaded_tokenizer.tokenize(sample_text)
word_ids = loaded_tokenizer.convert_tokens_to_ids(words)
word_tensor = torch.tensor([word_ids[:max_lenght]])

x = word_tensor.cuda()
y = loaded_model(x)
pred = y[0].argmax(-1)
print("result: ", dirs[pred])

result:  movie-enter
